# BERT for Sentiment Analysis => fine-tuning a pre-trained BERT

In [15]:
#%pip install transformers -U

In [16]:
import transformers
import tensorflow as tf

# Downloading large review movie dataset (25000 reviews)

In [17]:
!wget https://thome.isir.upmc.fr/classes/RITAL/json_pol.json

--2026-03-29 23:41:44--  https://thome.isir.upmc.fr/classes/RITAL/json_pol.json
Resolving thome.isir.upmc.fr (thome.isir.upmc.fr)... 134.157.18.247
Connecting to thome.isir.upmc.fr (thome.isir.upmc.fr)|134.157.18.247|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32663547 (31M) [application/json]
Saving to: ‘json_pol.json.2’

json_pol.json.2     100%[===================>]  31.15M  16.1MB/s    in 1.9s    

2026-03-29 23:41:46 (16.1 MB/s) - ‘json_pol.json.2’ saved [32663547/32663547]



In [18]:
import json
from collections import Counter

# Loading json
file = './json_pol.json'
with open(file,encoding="utf-8") as f:
    data = json.load(f)


# Quick Check
counter = Counter((x[1] for x in data))
print("Number of reviews : ", len(data))
print("----> # of positive : ", counter[1])
print("----> # of negative : ", counter[0])
print("")
print(data[0])


Number of reviews :  25000
----> # of positive :  12500
----> # of negative :  12500

['Although credit should have been given to Dr. Seuess for stealing the story-line of "Horton Hatches The Egg", this was a fine film. It touched both the emotions and the intellect. Due especially to the incredible performance of seven year old Justin Henry and a script that was sympathetic to each character (and each one\'s predicament), the thought provoking elements linger long after the tear jerking ones are over. Overall, superior acting from a solid cast, excellent directing, and a very powerful script. The right touches of humor throughout help keep a "heavy" subject from becoming tedious or difficult to sit through. Lastly, this film stands the test of time and seems in no way dated, decades after it was released.', 1]


# Getting the Tokenizer

In [19]:
#from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
#tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-cased')
model_name = "haisongzhang/roberta-tiny-cased"
#model_name = "bert-base-cased"

#model_name = "distilbert-base-uncased-finetuned-yelp-polarity"
#model_name = "textattack/bert-base-uncased-yelp-polarity"
#model_name = "rttl-ai/bert-base-uncased-yelp-reviews"


from transformers import AutoTokenizer#, BertForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Experiment the Tokenizer on the first train review

In [20]:
maxL = 512 # Max length of the sequence

string_tokenized = tokenizer(data[0][0], return_tensors="pt",
                                        add_special_tokens=True,  # add '[CLS]' and '[SEP]'
                            max_length=maxL,  # set max length
                            truncation=True,  # truncate longer messages
                            #pad_to_max_length=True
                            padding='max_length',  # add padding
                            return_attention_mask=True)

The output of the tokenizer string_tokenized (class BatchEncoding) returns two elements:


*   string_tokenized['input_ids']: the index of each token in the dictionary
*   string_tokenized['attention_mask']: a binary mask (0 to ignore the token, 1 to consider it). This is because we need tensor a fixed length and we have reviews with a variable number of words



In [21]:
print(string_tokenized['input_ids'][0][:25])
print(tokenizer.convert_ids_to_tokens(string_tokenized['input_ids'][0][:25]))
print(string_tokenized['attention_mask'][0][:25])

tensor([  101,  1780,  4755,  1431,  1138,  1151,  1549,  1106,   173,  1197,
          119, 14516, 10589,  1116,  1111, 11569,  1103,  1642,   118,  1413,
         1104,   107, 16358, 11530, 15524])
['[CLS]', 'although', 'credit', 'should', 'have', 'been', 'given', 'to', 'd', '##r', '.', 'se', '##ues', '##s', 'for', 'stealing', 'the', 'story', '-', 'line', 'of', '"', 'ho', '##rton', 'hatch']
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1])


**You can use the BERT model for directly predicting polarity.** Let us apply that on the first review which has been tokenized with string_tokenized.

# Let's tokenize the whole dataset

In [22]:
import numpy as np

maxL = 512


inputs_tokens = []
attention_masks = []

for i in range(len(data)):
    if(i%2500==0):
        print(i)
    string_tokenized = tokenizer(data[i][0], return_tensors="pt",
                                        add_special_tokens=True,  # add '[CLS]' and '[SEP]'
                            max_length=maxL,  # set max length
                            truncation=True,  # truncate longer messages
                            #pad_to_max_length=True
                            padding='max_length',  # add padding
                            return_attention_mask=True)

    inputs_tokens.append(string_tokenized['input_ids'])
    attention_masks.append(string_tokenized['attention_mask'])

0
2500
5000
7500
10000
12500
15000
17500
20000
22500


# Let's create a 'TensorDataSet' FOR THE SAMPLES where each element is a triplet composed of token word index, token mask, and label

In [23]:
print(data[1300][1])
print(inputs_tokens[1300].shape)
print(attention_masks[1300].shape)

1
torch.Size([1, 512])
torch.Size([1, 512])


In [24]:
import torch
# Converting input tokens to torch tensors
inputs_tokens = torch.cat(inputs_tokens, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)


In [25]:
# Converting labels to torch tensor
#y = torch.zeros((len(data),2), dtype=torch.float)
y = torch.zeros((len(data),), dtype=torch.float)

for i in range(len(data)):
    y[i] = data[i][1]
    #y[i][data[i][1]] = 1
#y = torch.from_numpy(y)

In [26]:
print(inputs_tokens.shape)
print(attention_masks.shape)
print(y.shape)

torch.Size([25000, 512])
torch.Size([25000, 512])
torch.Size([25000])


In [27]:
from sklearn.model_selection import train_test_split

np.random.seed(0)
rs=10

inputs_tokens_train, inputs_tokens_test, attention_masks_train, attention_masks_test, y_train, y_test =train_test_split(inputs_tokens, attention_masks, y, test_size=0.5, random_state=rs)

print(inputs_tokens_train.shape)
print(inputs_tokens_test.shape)

print(attention_masks_train.shape)
print(attention_masks_test.shape)

print(y_train.shape)
print(y_test.shape)

torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500, 512])
torch.Size([12500])
torch.Size([12500])


In [28]:
print(y_train[:10])
print("nb positive train =", int(y_train.sum().item()))
print("nb positive test =", int(y_test.sum().item()))

tensor([0., 1., 1., 0., 1., 0., 0., 1., 0., 1.])
nb positive train = 6237
nb positive test = 6263


In [29]:
from torch.utils.data import TensorDataset, random_split, DataLoader, RandomSampler, SequentialSampler

dataset_train = TensorDataset(inputs_tokens_train,  attention_masks_train, y_train)
dataset_test = TensorDataset(inputs_tokens_test,  attention_masks_test, y_test)

# Lets download a BERT model for word embedding

In [30]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 47685.44it/s]
BertForSequenceClassification LOAD REPORT from: haisongzhang/roberta-tiny-cased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 512, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): LayerNorm((512,), eps=1e-12, e

In [32]:
train_dataloader_tmp = DataLoader(dataset_train, batch_size=2,shuffle=True)

it, tm, labelsb = next(iter(train_dataloader_tmp))

pred = model(input_ids=it, attention_mask=tm)
print(pred.logits)

tensor([[-0.1544,  0.0571],
        [-0.1332,  0.0772]], grad_fn=<AddmmBackward0>)


In [33]:
print(pred.logits.argmax(axis=1))

tensor([1, 1])


# FINE-TUNING THE MODEL

In [34]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [35]:
def accuracy(model, dataloader):
  model.eval()
  nbgood = 0
  nball = 0
  for idx,batch in enumerate(dataloader):
    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)
    b_labels = batch[2].to(device).long()

    with torch.no_grad():
      pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
      yhat = pred.logits.argmax(axis=1)
      ytrue = b_labels
      nbgood += (yhat==ytrue).sum().item()
      nball += len(b_labels)

    if(idx>0 and idx%250==0):
      print("idx=",idx," nbgood=",nbgood)

  acc = nbgood / nball
  return acc


In [36]:
def accuracy_batch(model, b_input_ids, b_input_mask, b_labels):
  model.eval()
  b_labels = b_labels.long()

  with torch.no_grad():
    pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
    yhat = pred.logits.argmax(axis=1)
    ytrue = b_labels

    print(yhat.detach().cpu().numpy())
    print(ytrue.detach().cpu().numpy())

    acc = (yhat==ytrue).sum().item() / len(b_labels)

  return acc


In [37]:
# create DataLoaders with samplers

import torch.nn as nn
import torch.optim as optim
tb = int(25)
train_dataloader = DataLoader(dataset_train, batch_size=tb,shuffle=True)
test_dataloader = DataLoader(dataset_test, batch_size=tb,shuffle=False)
nbTrain = len(data)

temb = 768
temb = 512
#features = np.zeros((nbTrain, temb))
#nbtach = int(nbTrain/tb)
#print(f"nb batches={nbtach}")

#nbgood =torch.tensor(0.0).cuda()
nbepochs =5
loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Comuting CLS features
model.to(device)

for e in range(nbepochs):
    model.train()
    loss_epoch = 0.0
    for idx,batch in enumerate(train_dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device).long()
        
        optimizer.zero_grad()
        
        pred = model(input_ids=b_input_ids, attention_mask=b_input_mask)
        #print(b_labels)
        ce = loss(pred.logits,b_labels)
        ce.backward()
        optimizer.step()
        loss_epoch += ce.item()
        
        if(idx>0 and idx%250==0):
            print("** batch: ",idx," loss=",ce.item()," acc train=",accuracy(model,train_dataloader)," acc test=",accuracy(model,test_dataloader) )
    
    print("**** epoch",(e+1)," loss=",loss_epoch/len(train_dataloader)," acc train=",accuracy(model,train_dataloader)," acc test=",accuracy(model,test_dataloader) )



idx= 250  nbgood= 5750
idx= 250  nbgood= 5539
** batch:  250  loss= 0.2251547873020172  acc train= 0.91776  acc test= 0.88416


KeyboardInterrupt: 

In [ ]:
b_input_ids, b_input_mask, b_labels = next(iter(test_dataloader))
b_input_ids = b_input_ids.to(device)
b_input_mask = b_input_mask.to(device)
b_labels = b_labels.to(device)

print("accuracy on one test batch =", accuracy_batch(model, b_input_ids, b_input_mask, b_labels))
print("global train accuracy =", accuracy(model, train_dataloader))
print("global test accuracy =", accuracy(model, test_dataloader))
